# Kaggle vLLM + ngrok

Run cells in order. GPU: T4 x2.

Requires Kaggle secrets: `NGROK_AUTH_TOKEN`, `HF_TOKEN`.

When done, opencode.json `baseURL` = printed NGROK URL + `/v1`.

In [ ]:
%pip install -q vllm

In [ ]:
!pkill -9 -x ngrok || true
!sleep 1
!which ngrok || curl -sSL -o /tmp/ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -q -o /tmp/ngrok.zip -d /tmp || true
!mv -f /tmp/ngrok /usr/local/bin/ngrok || true

In [ ]:
!pkill -9 -if vllm || true
!pkill -9 -x ngrok || true
!fuser -k 8000/tcp 2>/dev/null || true
!sleep 2
!nvidia-smi

In [ ]:
import importlib
import json
import os
import subprocess
import time
import urllib.error
import urllib.request

try:
    kaggle_secrets = importlib.import_module("kaggle_secrets")
    UserSecretsClient = kaggle_secrets.UserSecretsClient
except ModuleNotFoundError as exc:
    raise RuntimeError("This cell must be run in a Kaggle notebook.") from exc

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

subprocess.run(
    ["ngrok", "config", "add-authtoken", secrets.get_secret("NGROK_AUTH_TOKEN")],
    check=True,
)
with open("/tmp/ngrok.log", "w") as ngrok_log:
    p = subprocess.Popen(
        ["ngrok", "http", "8000", "--log", "stdout"],
        stdout=ngrok_log,
        stderr=subprocess.STDOUT,
    )

for _ in range(60):
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:4040/api/tunnels", timeout=5
        ) as response:
            d = json.load(response)
        url = d["tunnels"][0]["public_url"]
        print("NGROK URL:", url)
        break
    except (urllib.error.URLError, json.JSONDecodeError, KeyError, IndexError):
        time.sleep(2)
else:
    with open("/tmp/ngrok.log") as ngrok_log:
        print(ngrok_log.read())

In [ ]:
MODEL = os.environ.get("VLLM_MODEL", "Qwen/Qwen2.5-7B-Instruct-AWQ")
TP = int(os.environ.get("VLLM_TP_SIZE", "2"))

subprocess.run(
    [
        "vllm",
        "serve",
        MODEL,
        "--served-model-name",
        MODEL,
        "--tensor-parallel-size",
        str(TP),
        "--max-model-len",
        "32768",
        "--gpu-memory-utilization",
        "0.90",
        "--enforce-eager",
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
        "--port",
        "8000",
    ],
    check=True,
)